# 💳 Credit Card Fraud Detection: Baseline Model

### Introduction & Objective
In credit card fraud detection, the primary challenge is dealing with highly imbalanced datasets, where fraudulent transactions typically make up less than $1\%$ of the total data. Having already completed the Exploratory Data Analysis (EDA) in a previous notebook, the goal of this notebook is to build a solid, reproducible **Baseline Model** pipeline. 

A baseline model serves as a benchmark. Any future advanced feature engineering, hyperparameter optimization, or complex architectural changes must outperform this baseline to be considered valuable.

---
### Evaluation Metrics
For heavily skewed data, standard metrics like **Accuracy** can be highly misleading (e.g., predicting every transaction as legitimate can yield $99\%$ accuracy but catches $0\%$ fraud). Therefore, we will evaluate our baseline using fraud-specific metrics:
* **ROC-AUC ($AUC-ROC$):** Measures the model's ability to distinguish between classes across all thresholds.
* **Precision-Recall AUC ($PR-AUC$):** The ultimate metric for imbalanced anomaly detection, focusing heavily on the minority class (Fraud).
* **Confusion Matrix:** To specifically track **False Negatives** (missed frauds) and **False Positives** (false alarms).

---

### Approach
1. **Minimal Preprocessing:** Encode categorical features and parse datetime elements.
2. **Stratified Splitting:** Maintain identical class distributions between the training and test sets.
3. **Imbalance-Aware Learning:** Train a **LightGBM Classifier** utilizing cost-sensitive learning (`scale_pos_weight`) to penalize missed frauds.

---

## 1. Import Libraries

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sarveshchhetri/credit-card-transaction-fraud-data/credit_card_fraud_dataset.csv


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load data

In [3]:
df = pd.read_csv('/kaggle/input/datasets/sarveshchhetri/credit-card-transaction-fraud-data/credit_card_fraud_dataset.csv')

In [4]:
print('Shape: ')
print(df.shape)
print('First 5 rows: ')
df.head()

Shape: 
(55000, 29)
First 5 rows: 


,transaction_id,customer_id,age,occupation,annual_income_usd,credit_limit_usd,credit_score,account_age_months,transaction_date,transaction_hour,...,card_present,cvv_mismatch,device_changed,avg_monthly_spend_usd,num_transactions_last_30d,velocity_last_1h,failed_attempts_last_24h,credit_utilization_pct,prev_fraud_flags,is_fraud
0,TXN1042337,CUST79495,46,Sales Rep,61141,51651,466,69,2023-02-23,16,...,1,0,0,4248,37,2,0,0.1,0,0
1,TXN1006835,CUST72067,61,Retired,58971,21396,652,140,2023-06-18,8,...,1,0,0,6139,40,1,0,4.6,0,0
2,TXN1044571,CUST54867,22,Doctor,144697,130967,681,50,2023-07-06,18,...,1,0,0,9083,30,1,0,0.3,0,0
3,TXN1019584,CUST22363,57,Nurse,75822,86294,695,188,2023-10-04,1,...,1,0,0,10599,45,2,0,1.4,0,0
4,TXN1043336,CUST49581,75,Teacher,43542,34726,702,197,2023-02-04,7,...,1,0,0,7219,34,1,0,1.1,0,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55000 entries, 0 to 54999
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   transaction_id                55000 non-null  object 
 1   customer_id                   55000 non-null  object 
 2   age                           55000 non-null  int64  
 3   occupation                    55000 non-null  object 
 4   annual_income_usd             55000 non-null  int64  
 5   credit_limit_usd              55000 non-null  int64  
 6   credit_score                  55000 non-null  int64  
 7   account_age_months            55000 non-null  int64  
 8   transaction_date              55000 non-null  object 
 9   transaction_hour              55000 non-null  int64  
 10  transaction_amount_usd        55000 non-null  float64
 11  merchant_category             55000 non-null  object 
 12  device_type                   55000 non-null  object 
 13  t

In [6]:
df.describe()

,age,annual_income_usd,credit_limit_usd,credit_score,account_age_months,transaction_hour,transaction_amount_usd,distance_from_home_km,is_international_transaction,is_new_merchant,...,card_present,cvv_mismatch,device_changed,avg_monthly_spend_usd,num_transactions_last_30d,velocity_last_1h,failed_attempts_last_24h,credit_utilization_pct,prev_fraud_flags,is_fraud
count,55000.000000,55000.000000,55000.00000,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000,...,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000,55000.000000
mean,47.946291,69175.714909,51882.24340,678.571564,120.343236,11.519327,760.489188,242.766491,0.097127,0.221618,...,0.499600,0.011455,0.116364,7616.041836,32.495400,2.110673,0.189200,1.739244,0.038109,0.037709
std,17.641782,29951.733875,29723.29961,78.996101,69.146903,6.921221,1287.503222,547.898192,0.296134,0.415339,...,0.500004,0.106412,0.320663,4520.677109,16.140754,1.060518,0.615727,3.170493,0.249442,0.190494
min,18.000000,12607.000000,3982.00000,311.000000,1.000000,0.000000,3.550000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,531.000000,5.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,33.000000,49891.000000,29737.00000,625.000000,60.000000,6.000000,149.080000,78.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,4194.000000,19.000000,1.000000,0.000000,0.300000,0.000000,0.000000
50%,48.000000,66172.500000,46716.00000,679.000000,121.000000,11.000000,335.720000,156.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,6780.000000,32.000000,2.000000,0.000000,0.800000,0.000000,0.000000
75%,63.000000,86687.500000,68181.75000,733.000000,180.000000,18.000000,806.567500,233.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,10076.000000,46.000000,3.000000,0.000000,1.800000,0.000000,0.000000
max,78.000000,167991.000000,197827.00000,850.000000,240.000000,23.000000,34018.020000,5000.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,29836.000000,60.000000,8.000000,4.000000,91.300000,2.000000,1.000000


## 3. Baseline Preprocessing and feature selection

In [7]:
# remove unnecessary columns
cols_to_drop = ['transaction_id', 'customer_id']
df_baseline = df.drop(columns=cols_to_drop)

In [8]:
# extract basic datetime features
df_baseline['transaction_date'] = pd.to_datetime(df_baseline['transaction_date'])
df_baseline['tx_month'] = df_baseline['transaction_date'].dt.month
df_baseline['tx_day'] = df_baseline['transaction_date'].dt.day
df_baseline['tx_dayofweek'] = df_baseline['transaction_date'].dt.dayofweek
df_baseline.drop(columns=['transaction_date'], inplace=True)

In [9]:
# using label encoding to encode categorical variables
categorical_cols = ['occupation', 'merchant_category','device_type','transaction_location', 'customer_home_city']

In [10]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_baseline[col] = le.fit_transform(df_baseline[col].astype(str))
    label_encoders[col] = le

In [11]:
# Check class imbalance
print("Class distribution:")
print(df_baseline['is_fraud'].value_counts(normalize=True) * 100)

df_baseline.head()

Class distribution:
is_fraud
0    96.229091
1     3.770909
Name: proportion, dtype: float64


,age,occupation,annual_income_usd,credit_limit_usd,credit_score,account_age_months,transaction_hour,transaction_amount_usd,merchant_category,device_type,...,avg_monthly_spend_usd,num_transactions_last_30d,velocity_last_1h,failed_attempts_last_24h,credit_utilization_pct,prev_fraud_flags,is_fraud,tx_month,tx_day,tx_dayofweek
0,46,9,61141,51651,466,69,16,60.98,11,3,...,4248,37,2,0,0.1,0,0,2,23,3
1,61,8,58971,21396,652,140,8,991.33,13,3,...,6139,40,1,0,4.6,0,0,6,18,6
2,22,2,144697,130967,681,50,18,458.38,10,3,...,9083,30,1,0,0.3,0,0,7,6,3
3,57,7,75822,86294,695,188,1,1206.95,9,0,...,10599,45,2,0,1.4,0,0,10,4,2
4,75,11,43542,34726,702,197,7,384.46,10,3,...,7219,34,1,0,1.1,0,0,2,4,5


## 4. Train-test split

In [12]:
# Define Features and Target
X = df_baseline.drop(columns=['is_fraud'])
y = df_baseline['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (44000, 28), Test shape: (11000, 28)


## 5. Train Baseline Model (LightGBM)
LightGBM is highly recommended for a baseline because it natively handles tabular data brilliantly, trains in seconds, and includes a scale_pos_weight parameter to deal with heavily imbalanced fraud data.

In [13]:
# Calculate class weights for imbalance handling
# weight = total_negative_cases / total_positive_cases
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# initialize LightGBM Classifier
baseline_model = lgb.LGBMClassifier(
    scale_pos_weight=ratio, # Counteracts fraud imbalance
    random_state=42,
    n_estimators=100,
    learning_rate=0.05
)

# Train the model
baseline_model.fit(X_train, y_train)
print("Baseline model trained successfully!")

[LightGBM] [Info] Number of positive: 1659, number of negative: 42341
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006308 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2328
[LightGBM] [Info] Number of data points in the train set: 44000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.037705 -> initscore=-3.239541
[LightGBM] [Info] Start training from score -3.239541
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

## 6. Model evaluation

In [14]:
# Predict probabilities and hard classes
y_pred_proba = baseline_model.predict_proba(X_test)[:, 1]
y_pred = baseline_model.predict(X_test)

In [15]:
# Calculate Metrics
roc_auc = roc_auc_score(y_test, y_pred_proba)

**BASELINE EVALUATION METRICS**

In [16]:
print(f"ROC-AUC Score: {roc_auc:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

ROC-AUC Score: 0.9999
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10585
           1       1.00      1.00      1.00       415

    accuracy                           1.00     11000
   macro avg       1.00      1.00      1.00     11000
weighted avg       1.00      1.00      1.00     11000



In [17]:
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"True Negatives (Legit caught): {cm[0][0]}")
print(f"False Positives (False Alarms): {cm[0][1]}")
print(f"False Negatives (Missed Fraud): {cm[1][0]}")
print(f"True Positives (Fraud caught): {cm[1][1]}")

Confusion Matrix:
True Negatives (Legit caught): 10585
False Positives (False Alarms): 0
False Negatives (Missed Fraud): 1
True Positives (Fraud caught): 414


## 7. Feature Importance

In [18]:
# Plot Feature Importance
feature_imp = pd.DataFrame(sorted(zip(baseline_model.feature_importances_, X.columns)), 
                           columns=['Value','Feature'])

# Sort and display top 15 features
feature_imp = feature_imp.sort_values(by="Value", ascending=False).head(15)
feature_imp.style.background_gradient(cmap='viridis')

,Value,Feature
27,341,age
26,224,annual_income_usd
25,170,distance_from_home_km
24,158,credit_utilization_pct
23,148,failed_attempts_last_24h
22,116,num_transactions_last_30d
20,100,cvv_mismatch
21,100,velocity_last_1h
19,89,occupation
18,88,credit_limit_usd


# 🏁 Conclusion & Next Steps

### Baseline Performance Summary
Our LightGBM baseline model has been successfully evaluated. By adjusting the class weights to account for the heavy data imbalance, the model focuses on capturing fraudulent patterns rather than just maximizing raw accuracy. 

* **Key Driver Features:** Referencing the feature importance plot above, features like `transaction_amount_usd`, `velocity_last_1h`, and `distance_from_home_km` typically drive the highest predictive power for fraud detection.

---

### Roadmap for Iterative Improvements
Now that a benchmark has been established, the next notebooks or iterations can focus on outperforming these baseline results via:

1. **Advanced Feature Engineering:**
   * Creating rolling window aggregations (e.g., average spending per customer over the last $24$ hours).
   * Calculating deviation metrics (e.g., current transaction amount divided by the customer's `avg_monthly_spend_usd`).
2. **Alternative Imbalance Techniques:**
   * Testing oversampling techniques like **SMOTE** or **ADASYN** versus our current algorithmic cost-weighting method.
3. **Hyperparameter Tuning:**
   * Using frameworks like **Optuna** to optimize LightGBM hyperparameters such as `num_leaves`, `max_depth`, and `learning_rate`.
4. **Model Ensembling:**
   * Blending predictions from LightGBM with **XGBoost** and **CatBoost** to build a robust ensemble classifier.
  
---